# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 44.5264


In [2]:
!pip show optimum-quanto

Name: optimum-quanto
Version: 0.2.6
Summary: A pytorch quantization backend for optimum.
Home-page: 
Author: David Corvoysier
Author-email: 
License: Apache-2.0
Location: /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/python3.10/site-packages
Requires: huggingface-hub, ninja, numpy, safetensors, torch
Required-by: 


In [3]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/hub/"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface/hub/
MemTotal: 251.59 GB
MemFree: 28.91 GB
MemAvailable: 247.37 GB
Free GPU Memory (GB): 44.5264

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA L40S

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



## 2. Loading Model

In [4]:
from transformers import AutoTokenizer

model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "openai-community/gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
# TODO: Check why dtype = auto solved the problem
# TODO: what is the default value of torch_dtype -> look in the githubb documentation
# Always use "auto"
model.NAME = model_name

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

### Load QUANTO model

In [ ]:
import logging
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from accelerate import init_empty_weights
from safetensors.torch import load_file
from optimum.quanto import requantize, qint8, qint4

from src import MODEL_SAVE_PATH

# Setup basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    # Set your paths here
    model_path = os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-QUANTO-8")
    base_model_path = "meta-llama/Meta-Llama-3-8B"
    state_dict_path = os.path.join(model_path, "model.safetensors")
    quantization_map_file = os.path.join(model_path, "quantization_map.json")
    
    weights = qint8
    activations = qint8
    
    # Load quantized weights
    state_dict = load_file(state_dict_path)
    with open(quantization_map_file, 'r') as f:
        quantization_map = json.load(f)
    
    model_reloaded = AutoModelForCausalLM.from_pretrained(base_model_path, trust_remote_code=True)
    config = AutoConfig.from_pretrained(base_model_path, trust_remote_code=True)
    with init_empty_weights():
        model_reloaded = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
    requantize(model_reloaded, state_dict, quantization_map, device)
    
    model = model_reloaded
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_path, device_map="cuda")
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    
    # Quick test
    test_input = "Hello, world! What is the location of Simcoe?"
    encoded = tokenizer(test_input, return_tensors="pt").to("cuda")
    output = model.generate(**encoded, max_new_tokens=20)
    print("Test output:", tokenizer.decode(output[0], skip_special_tokens=True))

except Exception as e:
    logger.error(f"Error loading model: {str(e)}")
    raise

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Test output: Hello, world! What is the location of Simcoe?щинаREC in.Formsonta..
 結弱.swing://.swing://://://.Formsagnost Commentary://://gements


In [5]:
import logging
import torch
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from accelerate import init_empty_weights
from safetensors.torch import load_file
from optimum.quanto import requantize, qint8, qint4
from src import MODEL_SAVE_PATH

# Setup basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_quantized_model(model_path, base_model_path, device="cuda"):
    try:
        # Set paths
        state_dict_path = os.path.join(model_path, "model.safetensors")
        quantization_map_file = os.path.join(model_path, "quantization_map.json")
        
        # Load quantization configuration
        logger.info("Loading quantization map...")
        with open(quantization_map_file, 'r') as f:
            quantization_map = json.load(f)
        
        # Load state dict
        logger.info("Loading state dictionary...")
        state_dict = load_file(state_dict_path)
        
        # Load configuration
        logger.info("Loading model configuration...")
        config = AutoConfig.from_pretrained(
            base_model_path,
            trust_remote_code=True,
            torch_dtype=torch.float16  # Ensure correct dtype
        )
        
        # Initialize empty model
        logger.info("Initializing empty model...")
        with init_empty_weights():
            model = AutoModelForCausalLM.from_config(
                config,
                trust_remote_code=True
            )
        
        # Requantize model
        logger.info("Requantizing model...")
        requantize(
            model,
            state_dict,
            quantization_map,
            device=device
        )
        
        # Move model to device
        model = model.to(device)
        
        return model
        
    except Exception as e:
        logger.error(f"Error in load_quantized_model: {str(e)}")
        raise

def load_tokenizer(base_model_path):
    try:
        logger.info("Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(
            base_model_path,
            trust_remote_code=True
        )
        
        # Set padding configuration
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        tokenizer.padding_side = "left"
        
        return tokenizer
        
    except Exception as e:
        logger.error(f"Error in load_tokenizer: {str(e)}")
        raise

def generate_text(model, tokenizer, prompt, max_new_tokens=20):
    try:
        # Encode input
        encoded = tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512  # Add reasonable max length
        ).to(model.device)
        
        # Generate with better parameters
        with torch.no_grad():
            output_ids = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=True,  # Enable sampling
                temperature=0.7,  # Add temperature
                top_p=0.95,      # Add top-p sampling
                top_k=50,        # Add top-k sampling
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1  # Prevent repetition
            )
        
        # Decode output
        generated_text = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        
        return generated_text
        
    except Exception as e:
        logger.error(f"Error in generate_text: {str(e)}")
        raise

if __name__ == "__main__":
    # Set your paths
    model_path = os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-QUANTO-4")
    base_model_path = "meta-llama/Meta-Llama-3-8B"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    try:
        # Load model and tokenizer
        model = load_quantized_model(model_path, base_model_path, device)
        tokenizer = load_tokenizer(base_model_path)
        
        # Test generation
        test_input = "Hello, world! What is the location of Simcoe?"
        generated_text = generate_text(model, tokenizer, test_input)
        logger.info(f"Input: {test_input}")
        logger.info(f"Generated text: {generated_text}")
        
    except Exception as e:
        logger.error(f"Error in main: {str(e)}")
        raise

INFO:__main__:Loading quantization map...
INFO:__main__:Loading state dictionary...
INFO:__main__:Loading model configuration...
INFO:__main__:Initializing empty model...
INFO:__main__:Requantizing model...
/nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/python3.10/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
ERROR:__main__:Error in load_quantized_model: Error building extension 'quanto_cuda': [1/7] /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/bin/nvcc --generate-dependencies-with-compile --dependency-output unpack.cuda.o.d -DTORCH_EXTENSION_NAME=quanto_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/p

RuntimeError: Error building extension 'quanto_cuda': [1/7] /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/bin/nvcc --generate-dependencies-with-compile --dependency-output unpack.cuda.o.d -DTORCH_EXTENSION_NAME=quanto_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/python3.10/site-packages/torch/include -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/python3.10/site-packages/torch/include/TH -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/lib/python3.10/site-packages/torch/include/THC -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/include -isystem /nfs/students/daro/miniconda3/envs/env-quant-rel-310-11-01/include/py

## 3. Loading Datasets

### 3.1. WikiText

In [5]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=256,
  tokenizer_name=model_name,
  seed=3,
)

wikitext_dataloader = wikitext_data_module.val_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up WikiTextDataModule...
################################


################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622

Length of total validation dataset (characters): 1142150
Length of tokenized validation dataset (tokens): 252726
Tokenizer compression rate: 22.13%

Number of batches in validation dataloader: 493

Batch 1:
  Original Text: 2011, Michigan beat Oakland 90 – 80, its highest @-@ scoring game since beating Northern Michigan 97 @-@ 50 on November 14, 2009. It was also Michigan's first game since 2002 with three 20 @-@ point scorers ( Hardaway, Burke and Evan Smotrycz ). Burke earned his second Freshman of the Week honor on December 12 after scoring a season @-@ high 20 poin

### 3.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=1
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 4. Quantization

### 4.4 Quanto

### 4.4.1 HuggingFace Integration - Won't work!!!

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

from src import MODEL_SAVE_PATH

model_name = "meta-llama/Meta-Llama-3-8B"
tokenizer_instance = AutoTokenizer.from_pretrained(model_name)
tokenizer_instance.pad_token_id = tokenizer_instance.eos_token_id

quantization_config = QuantoConfig(weights="int8", activations="int8")

quanto_model_1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype="auto",
    device_map="cuda"
)

# Set the model's PATH and NAME attributes
quanto_model_1_name = f"{model_name.split('/')[1]}-QUANTO-1"
quanto_model_1_path = os.path.join(MODEL_SAVE_PATH, quanto_model_1_name)
quanto_model_1.PATH = quanto_model_1_path
quanto_model_1.NAME = quanto_model_1_name

# Save the quantized model using `safetensors`
from safetensors.torch import save_file

# Save the model state dictionary
safe_file_path = f"{quanto_model_1_path}.safetensors"
save_file(quanto_model_1.state_dict(), safe_file_path)

# Save the quantization map to a JSON file
import json
from optimum.quanto import quantization_map

quantization_map_path = f"{quanto_model_1_path}_quantization_map.json"
with open(quantization_map_path, 'w') as f:
    json.dump(quantization_map(quanto_model_1), f)

print(f"Quantized model saved at {safe_file_path}")
print(f"Quantization map saved at {quantization_map_path}")

### 4.4.2 Optimum Quanto - No Finetuning

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file, save_file

from optimum.quanto import quantize, freeze, qint8, qfloat8, qint4
from src import MODEL_SAVE_PATH

# Load the model and tokenizer
quanto_model_2 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer_instance = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
tokenizer_instance.pad_token_id = tokenizer_instance.eos_token_id

# Quantize the model
quanto_model_2.eval()
weights = qint4
activations = None
quantize(quanto_model_2, weights=weights, activations=activations)

# Freeze the model to convert weights to integers
freeze(quanto_model_2)

quanto_model_2_name = f"{model_name.split('/')[1]}-QUANTO-4"
quanto_model_2_path = os.path.join(MODEL_SAVE_PATH, quanto_model_2_name)
os.makedirs(quanto_model_2_path, exist_ok=True)
safetensors_file = os.path.join(quanto_model_2_path, "model.safetensors")
quantization_map_file = os.path.join(quanto_model_2_path, "quantization_map.json")

quanto_model_2.PATH = quanto_model_2_path
quanto_model_2.NAME = quanto_model_2_name

# Save the quantized model
save_file(quanto_model_2.state_dict(), safetensors_file)

import json
from optimum.quanto import quantization_map

with open(quantization_map_file, 'w') as f:
  json.dump(quantization_map(quanto_model_2), f)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### 4.4.3 Optimum Quanto - Calibration

In [6]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 44.5244


In [19]:
!nvidia-smi

Sat Nov  2 23:29:44 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.08             Driver Version: 535.161.08   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L40S                    On  | 00000000:02:00.0 Off |                    0 |
| N/A   36C    P0              81W / 350W |  32586MiB / 46068MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [7]:
import torch
from tqdm import tqdm  # For progress bar
from safetensors.torch import load_file, save_file

from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.quanto import quantize, freeze, qint8, qfloat8, qint4, Calibration
from src import MODEL_SAVE_PATH

# Load the model and tokenizer
quanto_model_3 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer_instance = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
tokenizer_instance.pad_token_id = tokenizer_instance.eos_token_id

# Load the calibration dataset
cal_dataloader = wikitext_dataloader

# Quantize the model
quanto_model_3.eval()
weights = qint4
activations = None
quantize(quanto_model_3, weights=weights, activations=activations)
# qfloat8 vs qint8 -> weight quantization is easier than activations -> maybe not quantizing activations at all?

# Calibrate the model
max_batches = 128  # Set to lower values for debugging
batch_size = 1
with Calibration(streamline=True, debug=False):
    quanto_model_3.eval()
    with torch.no_grad():
        pbar = tqdm(wikitext_dataloader, total=min(max_batches, len(wikitext_dataloader)) if max_batches else len(wikitext_dataloader))
        for batch_idx, (input_ids, target_ids) in enumerate(pbar):
            # Optional early stopping
            if max_batches and batch_idx >= max_batches:
                break
            print(f"Calibrating with batch {batch_idx + 1}/{max_batches}")
            input_ids = input_ids.to("cuda")
            attention_mask = torch.ones_like(input_ids, dtype=torch.float)
            
            if batch_idx == 0:
                print(f"Input shape: {input_ids.shape}")
                print(f"Mask shape: {attention_mask.shape}")
                print(f"Sample mask: {attention_mask[0][:50]}")  # Show first 50 values
            
            quanto_model_3(input_ids)
        pbar.close()

# Freeze the model to convert weights to integers
freeze(quanto_model_3)

quanto_model_3_name = f"{model_name.split('/')[1]}-QUANTO-CALIB-4"
quanto_model_3_path = os.path.join(MODEL_SAVE_PATH, quanto_model_3_name)
os.makedirs(quanto_model_3_path, exist_ok=True)
safetensors_file = os.path.join(quanto_model_3_path, "model.safetensors")
quantization_map_file = os.path.join(quanto_model_3_path, "quantization_map.json")

quanto_model_3.PATH = quanto_model_3_path
quanto_model_3.NAME = quanto_model_3_name

# Save the quantized model
save_file(quanto_model_3.state_dict(), safetensors_file)

import json
from optimum.quanto import quantization_map

with open(quantization_map_file, 'w') as f:
    json.dump(quantization_map(quanto_model_3), f)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/128 [00:00<?, ?it/s]

Calibrating with batch 1/128
Input shape: torch.Size([1, 2048])
Mask shape: torch.Size([1, 2048])
Sample mask: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       device='cuda:0')


  1%|          | 1/128 [00:03<06:37,  3.13s/it]

Calibrating with batch 2/128


...

 98%|█████████▊| 126/128 [03:39<00:03,  1.76s/it]

Calibrating with batch 127/128


 99%|█████████▉| 127/128 [03:41<00:01,  1.76s/it]

Calibrating with batch 128/128


100%|██████████| 128/128 [03:43<00:00,  1.75s/it]


### 4.4.4 Optimum Quanto - QAT

In [5]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 44.5244


In [ ]:
model_name

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.quanto import quantize, freeze, qint8, qfloat8, qint4, QTensor
from torch.optim import Adam
from src import MODEL_SAVE_PATH

# Load the model and tokenizer
print("Loading the model and tokenizer...")
quanto_model_4 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
tokenizer.pad_token_id = tokenizer.eos_token_id

# Quantize the model
print("Quantizing the model...")
weights = qint8
activations = None
quantize(quanto_model_4, weights=qint8, activations=qint8)

# Quantization-Aware Training (QAT)
print("Quantization-Aware Training (QAT)...")
train_samples = 10
train_dataloader = wikitext_data_module.val_dataloader()
quanto_model_4.train()
optimizer = Adam(quanto_model_4.parameters(), lr=1e-4)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

n_epochs = 1  # Set lower values for debugging
for epoch in range(n_epochs):
    for batch_idx, (data, target) in enumerate(train_dataloader):
        print(f"Processing batch {batch_idx + 1}/{train_samples}")
        print("Memory before model:")
        !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
        if batch_idx >= train_samples:
            break
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            output = quanto_model_4(data)
            print("Memory after model:")
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            logits = output.logits
            if isinstance(logits, QTensor):
                print("Dequantizing logits...")
                logits = logits.dequantize()
            loss = torch.nn.functional.nll_loss(logits.view(1, logits.shape[2], logits.shape[1]), target)
            loss.backward()
            optimizer.step()
            
# Freeze the model to convert weights to integers
print("Freezing the model...")
freeze(quanto_model_4)

quanto_model_4_name = f"{model_name.split('/')[1]}-QUANTO-QAT-8"
quanto_model_4_path = os.path.join(MODEL_SAVE_PATH, quanto_model_4_name)
os.makedirs(quanto_model_4_path, exist_ok=True)
safetensors_file = os.path.join(quanto_model_4_path, "model.safetensors")
quantization_map_file = os.path.join(quanto_model_4_path, "quantization_map.json")

quanto_model_4.PATH = quanto_model_4_path
quanto_model_4.NAME = quanto_model_4_name

# Save the quantized model
save_file(quanto_model_4.state_dict(), safetensors_file)

import json
from optimum.quanto import quantization_map

with open(quantization_map_file, 'w') as f:
    json.dump(quantization_map(quanto_model_4), f)

Loading the model and tokenizer...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Quantizing the model...
Quantization-Aware Training (QAT)...
Free GPU Memory (GB): 27.8369
Processing batch 1/10
Memory before model:
Free GPU Memory (GB): 27.8369


/tmp/ipykernel_3842913/2961109856.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 44.53 GiB of which 1.25 MiB is free. Including non-PyTorch memory, this process has 44.52 GiB memory in use. Of the allocated memory 43.94 GiB is allocated by PyTorch, and 88.71 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
loss

In [ ]:
# Freeze the model to convert weights to integers
print("Freezing the model...")
freeze(quanto_model_4)

quanto_model_4_name = f"{model_name.split('/')[1]}-QUANTO-4"
quanto_model_4_path = os.path.join(MODEL_SAVE_PATH, quanto_model_4_name)
quanto_model_4.PATH = quanto_model_4_path
quanto_model_4.NAME = quanto_model_4_name

# Save the quantized model
print("Saving the quantized model...")
safe_save(quanto_model_4.state_dict(), f"{quanto_model_4_path}.safetensors")

### 4.5 Optimum Quanto - Old Version

In [ ]:
import logging
import torch
import io
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from accelerate import init_empty_weights
from safetensors.torch import load_file
from optimum.quanto import quantize, freeze

# Setup basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    # Set your paths here
    model_path = os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-QUANTO")
    base_model_path = "meta-llama/Meta-Llama-3-8B"
    state_dict_path = f"{model_path}.safetensors"
    
    weights = QUANTO_ACTIV_QUANT.get("weights")
    activations = QUANTO_ACTIV_QUANT.get("activations")
    
    logger.info("Loading QUANTO quantized model...")
    
    # Create empty model from config
    config = AutoConfig.from_pretrained(base_model_path, trust_remote_code=True)
    with init_empty_weights():
        model = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
    
    # Quantize model
    quantize(model, weights=weights, activations=activations)
    
    # Freeze model
    freeze(model)
    
    # Verify serialization
    logger.info("Verifying serialization...")
    b = io.BytesIO()
    torch.save(model.state_dict(), b)
    b.seek(0)
    state_dict = torch.load(b)
    
    # Reload model
    model_reloaded = AutoModelForCausalLM.from_config(config, trust_remote_code=True)
    quantize(model_reloaded, weights=state_dict, activations=None)
    model_reloaded.load_state_dict(state_dict)
    model = model_reloaded
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, device_map="cuda")
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    
    # Quick test
    test_input = "Hello, world!"
    encoded = tokenizer(test_input, return_tensors="pt").to("cuda")
    output = model.generate(**encoded, max_new_tokens=20)
    print("Test output:", tokenizer.decode(output[0], skip_special_tokens=True))

except Exception as e:
    logger.error(f"Error loading model: {str(e)}")
    raise

## 5. Evaluation

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

### 5.1. Perplexity

In [ ]:
quanto_model_2.to("cuda")

In [ ]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# Evaluate Perplexity
print("\n################################")
print("Evaluating Perplexity...")
print("################################\n")

def evaluate_perplexity(model, dataloader, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Metric on current batch
            perplexity = metric(logits.dequantize(), y)
            print(f"Perplexity: {perplexity:.2f}")

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"\nFinal Perplexity (PPL): {perplexity:.3f}")
    return perplexity.item()

wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(quanto_model_2, wikitext_dataloader, device="cuda")
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(quantized_model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 5.2. Brier Score

In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

## 6. Plotting activation values

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import os
import numpy as np
from optimum.quanto import quantize, freeze, qint8, qfloat8, safe_save

# Helper functions to register hooks and extract activations
def get_activation(name, activations):
    def hook(model, input, output):
        if isinstance(output, tuple):
            output = output[0]
        activations[name] = output.detach()
    return hook

def plot_activations_pair(layer_name, activations_original, activations_quanto):
    activation_values_original = activations_original[layer_name].cpu().numpy()
    activation_values_quanto = activations_quanto[layer_name].cpu().numpy()

    # Calculate percentiles for original model
    min_val_orig = np.min(activation_values_original)
    max_val_orig = np.max(activation_values_original)
    p1_orig = np.percentile(activation_values_original, 1)
    p99_orig = np.percentile(activation_values_original, 99)
    p25_orig = np.percentile(activation_values_original, 25)
    p75_orig = np.percentile(activation_values_original, 75)

    # Calculate percentiles for quantized model
    min_val_quanto = np.min(activation_values_quanto)
    max_val_quanto = np.max(activation_values_quanto)
    p1_quanto = np.percentile(activation_values_quanto, 1)
    p99_quanto = np.percentile(activation_values_quanto, 99)
    p25_quanto = np.percentile(activation_values_quanto, 25)
    p75_quanto = np.percentile(activation_values_quanto, 75)

    # Plot histogram with log scale for both models
    plt.figure(figsize=(20, 6))

    # Original model histogram
    plt.subplot(1, 2, 1)
    plt.hist(activation_values_original.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_orig, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_orig, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_orig, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_orig, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_orig, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_orig, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Original)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Quantized model histogram
    plt.subplot(1, 2, 2)
    plt.hist(activation_values_quanto.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_quanto, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_quanto, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_quanto, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_quanto, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_quanto, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_quanto, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Quantized)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Save the histograms
    os.makedirs('plots/activations_quanto', exist_ok=True)
    plt.savefig(f"plots/activations_quanto/{layer_name}_histogram.png")
    plt.close()

    # Plot activations in original order with percentile bands for both models
    plt.figure(figsize=(20, 6))

    # Original model activations
    plt.subplot(1, 2, 1)
    plt.plot(activation_values_original.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_original.flatten())), p25_orig, p75_orig, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_original.flatten())), p1_orig, p99_orig, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_original.flatten(), min_val_orig), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_original.flatten(), max_val_orig), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Original)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Quantized model activations
    plt.subplot(1, 2, 2)
    plt.plot(activation_values_quanto.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p25_quanto, p75_quanto, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p1_quanto, p99_quanto, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_quanto.flatten(), min_val_quanto), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_quanto.flatten(), max_val_quanto), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Quantized)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Save the ordered activations
    plt.savefig(f"plots/activations_quanto/{layer_name}_ordered.png")
    plt.close()

# Load the GPT-2 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "TinyLlama/TinyLlama_v1.1"
print(f"Loading model: {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")

# Example input
input_text = "The quick brown fox"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Extract activations for the original model
activations_original = {}
print("Registering hooks for the original model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_original))

# Perform a forward pass
print("Performing forward pass for the original model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Quantize the model using QUANTO
print("Quantizing the model using QUANTO...")
quantize(model, weights=qint8, activations=qint8)
freeze(model)

# Extract activations for the quantized model
activations_quanto = {}
print("Registering hooks for the quantized model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_quanto))

# Perform a forward pass with the quantized model
print("Performing forward pass for the quantized model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Plot activations for both the original and quantized models
print("Plotting activations for both models...")
for layer_name in activations_original.keys():
    plot_activations_pair(layer_name, activations_original, activations_quanto)

print("Plots saved for original and quantized models.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import os
import numpy as np
from optimum.quanto import quantize, freeze, qint8, qfloat8, safe_save

# Helper functions to register hooks and extract activations
def get_activation(name, activations):
    def hook(model, input, output):
        if isinstance(output, tuple):
            output = output[0]
        activations[name] = output.detach()
    return hook

def plot_activations_pair(layer_name, activations_original, activations_quanto):
    activation_values_original = activations_original[layer_name].cpu().numpy()
    activation_values_quanto = activations_quanto[layer_name].cpu().numpy()

    # Calculate percentiles for original model
    min_val_orig = np.min(activation_values_original)
    max_val_orig = np.max(activation_values_original)
    p1_orig = np.percentile(activation_values_original, 1)
    p99_orig = np.percentile(activation_values_original, 99)
    p25_orig = np.percentile(activation_values_original, 25)
    p75_orig = np.percentile(activation_values_original, 75)

    # Calculate percentiles for quantized model
    min_val_quanto = np.min(activation_values_quanto)
    max_val_quanto = np.max(activation_values_quanto)
    p1_quanto = np.percentile(activation_values_quanto, 1)
    p99_quanto = np.percentile(activation_values_quanto, 99)
    p25_quanto = np.percentile(activation_values_quanto, 25)
    p75_quanto = np.percentile(activation_values_quanto, 75)

    # Plot histogram with log scale for both models
    plt.figure(figsize=(20, 6))

    # Original model histogram
    plt.subplot(1, 2, 1)
    plt.hist(activation_values_original.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_orig, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_orig, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_orig, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_orig, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_orig, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_orig, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Original)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Quantized model histogram
    plt.subplot(1, 2, 2)
    plt.hist(activation_values_quanto.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_quanto, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_quanto, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_quanto, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_quanto, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_quanto, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_quanto, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Quantized)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Save the histograms
    os.makedirs('plots/activations_quanto', exist_ok=True)
    plt.savefig(f"plots/activations_quanto/{layer_name}_histogram.png")
    plt.close()

    # Plot activations in original order with percentile bands for both models
    plt.figure(figsize=(20, 6))

    # Original model activations
    plt.subplot(1, 2, 1)
    plt.plot(activation_values_original.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_original.flatten())), p25_orig, p75_orig, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_original.flatten())), p1_orig, p99_orig, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_original.flatten(), min_val_orig), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_original.flatten(), max_val_orig), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Original)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Quantized model activations
    plt.subplot(1, 2, 2)
    plt.plot(activation_values_quanto.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p25_quanto, p75_quanto, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p1_quanto, p99_quanto, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_quanto.flatten(), min_val_quanto), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_quanto.flatten(), max_val_quanto), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Quantized)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Save the ordered activations
    plt.savefig(f"plots/activations_quanto/{layer_name}_ordered.png")
    plt.close()

# Load the GPT-2 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "TinyLlama/TinyLlama_v1.1"
print(f"Loading model: {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")

# Example input
input_text = "The quick brown fox"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Extract activations for the original model
activations_original = {}
print("Registering hooks for the original model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_original))

# Perform a forward pass
print("Performing forward pass for the original model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Quantize the model using QUANTO
print("Quantizing the model using QUANTO...")
quantize(model, weights=qint8, activations=qint8)
freeze(model)

# Extract activations for the quantized model
activations_quanto = {}
print("Registering hooks for the quantized model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_quanto))

# Perform a forward pass with the quantized model
print("Performing forward pass for the quantized model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Plot activations for both the original and quantized models
print("Plotting activations for both models...")
for layer_name in activations_original.keys():
    plot_activations_pair(layer_name, activations_original, activations_quanto)

print("Plots saved for original and quantized models.")